# 04g. 포함 관계 암종 쌍 2단계 판정 (KIPAN⊃KIRC, GBMLGG⊃LGG)

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 제출 11 구성(3모델 균등)의 확률은 그대로 두고, 두 쌍 안에서의 최종 선택만 작은 전용 모델로 바꿨을 때 Macro F1 변화 확인
- 배경: 변이가 같고 라벨만 다른 쌍둥이가 학습과 평가로 갈리면 큰 모델은 학습 쪽 라벨을 외워서 틀림 (notebooks/H_data/08_twin_fold_comparison.ipynb)
- 방법: 쌍 확률을 더해 다른 암종과 비교 → 쌍이 1등이면 쌍 환자만으로 학습한 작은 LR(유전자 10개 + 변이 수)로 선택
- 평가: 팀 고정 홀드아웃의 개발셋(4,960명) 5-fold → 홀드아웃(1,241명) 1회 → 쌍둥이 분리 fold(개발셋, LR E 단독)
- 입력: experiments/preds의 OOF·test 예측 (XGBoost·LightGBM 재학습 없음)
- 규정: 모든 fit은 학습 행만 사용, test는 행별 변환 후 예측에만 사용, 같은 행을 찾아 라벨을 바꾸는 처리 없음
- 판단 기준 (팀 공지 11/12): 기존 3모델 대비 OOF +0.005 이상이면 제출 후보
- 기록표에는 적지 않음 (결과는 이 노트북 출력), Run All 가능 (약 3분)

## 0. 환경 설정 (수정하지 말고 그대로 실행)

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


## 1. 도구·기준 모델

In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

from src.config import EXPERIMENT_DIR
from src.data_loader import load_data, split_xy, encode_target
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags
from src.preprocessing import binarize_mutations
from src.validation import get_folds, twin_groups
from src.prediction import make_submission
from scripts.fixed_holdout_manifest import verify

# 제출 11 구성: 3모델 균등 (팀 공통 그룹 fold OOF, test는 train 전체 재학습)
BASE_KEYS = [
    "A_model__12_weighted_xgb_E_cpu",          # 가중치 XGBoost (E, CPU)
    "A_model__12_lr_position_bin50",           # LR (E + 위치 구간 50)
    "A_model__lgbm_g3_recomputed_for_B_11",    # LightGBM g3
]
PAIRS = (("KIPAN", "KIRC"), ("GBMLGG", "LGG"))  # (넓은 암종, 그 안에 포함되는 암종)
STAGE2_PARAMS = {"k": 10, "C": 0.05}            # 2단계 모델 크기: 유전자 10개, 강한 규제

## 2. 2단계 판정 함수 — 작은 LR(Pipeline 안에서 유전자 선택), 쌍 확률 합산 후 선택

In [3]:
def simple_features(X):
    """2단계 입력: 유전자별 변이 여부(0/1) + log(변이 유전자 수). 한 행 안에서만 계산."""
    b = binarize_mutations(X)
    return csr_matrix(np.c_[b.to_numpy(np.float32), np.log1p(b.sum(axis=1).to_numpy(np.float32))])


def make_stage2(n_genes):
    """유전자 k개 선택(chi2) + 변이 수 → 작은 LR. 선택은 Pipeline 안에서 학습 행에만 fit."""
    select = ColumnTransformer([
        ("genes", SelectKBest(chi2, k=STAGE2_PARAMS["k"]), slice(0, n_genes)),
        ("count", "passthrough", [n_genes]),
    ])
    lr = LogisticRegression(C=STAGE2_PARAMS["C"], class_weight="balanced", max_iter=2000, random_state=SEED)
    return Pipeline([("select", select), ("model", lr)])


def fit_stage2(F, y, rows, ia, ib):
    """rows 중 쌍(ia, ib) 환자만으로 2단계 모델 학습. 타깃: ib(포함되는 암종)이면 1."""
    rows = rows[np.isin(y[rows], [ia, ib])]
    return make_stage2(F.shape[1] - 1).fit(F[rows], (y[rows] == ib).astype(int))


def stage2_proba(F_fit, y_fit, fit_rows, F_apply, classes):
    """쌍마다 2단계 모델을 fit_rows로 학습해 F_apply 각 행의 '포함되는 암종' 확률 반환."""
    out = {}
    for a, b in PAIRS:
        model = fit_stage2(F_fit, y_fit, fit_rows, classes.index(a), classes.index(b))
        out[(a, b)] = model.predict_proba(F_apply)[:, 1]
    return out


def decode(proba, s2, classes):
    """쌍 확률을 더해 다른 암종과 비교 → 쌍 안에서는 2단계 확률(0.5 기준)로 선택."""
    out = proba.copy()
    for a, b in PAIRS:
        ia, ib = classes.index(a), classes.index(b)
        pair_sum = proba[:, ia] + proba[:, ib]
        pick_b = s2[(a, b)] > 0.5
        out[:, ia] = np.where(pick_b, 0, pair_sum)
        out[:, ib] = np.where(pick_b, pair_sum, 0)
    return out.argmax(1)


mf = lambda yt, pred: f1_score(yt, pred, average="macro")

## 3. 데이터·저장 예측 불러오기

In [4]:
train, test = load_data(DATA_DIR)   # test는 마지막 예측 파일에만 사용
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
classes = list(le.classes_)

dev_idx, hold_idx = verify()                     # 팀 고정 홀드아웃 (새로 만들지 않음)
X_dev_raw = X.iloc[dev_idx].reset_index(drop=True)
y_dev, y_hold = y[dev_idx], y[hold_idx]
folds_dev = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)

pred_dir = EXPERIMENT_DIR / "preds"
oof = sum(np.load(pred_dir / f"{k}__oof.npy") for k in BASE_KEYS) / len(BASE_KEYS)
test_proba = sum(np.load(pred_dir / f"{k}__test.npy") for k in BASE_KEYS) / len(BASE_KEYS)
oof_dev, oof_hold = oof[dev_idx], oof[hold_idx]

F = simple_features(X)
F_dev, F_hold = F[dev_idx], F[hold_idx]
print("개발", len(dev_idx), "| 홀드아웃", len(hold_idx), "| 2단계 입력 열", F.shape[1])

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
개발 4960 | 홀드아웃 1241 | 2단계 입력 열 4385


## 4. 개발셋 교차 평가 — fold별 변화

In [5]:
# fold 학습 부분으로 2단계 모델을 학습하고 남은 fold에 적용 × 5회
pred_base = oof_dev.argmax(1)
pred_new = np.zeros(len(y_dev), dtype=int)
for tr, va in folds_dev:
    s2 = stage2_proba(F_dev, y_dev, tr, F_dev[va], classes)
    pred_new[va] = decode(oof_dev[va], s2, classes)

fold_table = pd.DataFrame([
    {"fold": i + 1, "기준": mf(y_dev[va], pred_base[va]), "2단계 판정": mf(y_dev[va], pred_new[va])}
    for i, (_, va) in enumerate(folds_dev)])
fold_table["변화"] = fold_table["2단계 판정"] - fold_table["기준"]
print(fold_table.round(4).to_string(index=False))
print("상승 fold 수:", (fold_table["변화"] > 0).sum(), "/ 5")

 fold     기준  2단계 판정      변화
    1 0.5276  0.5274 -0.0002
    2 0.5335  0.5323 -0.0013
    3 0.5213  0.5232  0.0019
    4 0.5162  0.5096 -0.0066
    5 0.5302  0.5249 -0.0052
상승 fold 수: 1 / 5


## 5. 결과 — 개발 CV · 홀드아웃 1회 · 암종별 F1

In [6]:
# 개발셋 전체로 2단계 모델 학습 → 홀드아웃 1회 평가
s2_hold = stage2_proba(F_dev, y_dev, np.arange(len(y_dev)), F_hold, classes)
hold_base, hold_new = oof_hold.argmax(1), decode(oof_hold, s2_hold, classes)

result = pd.DataFrame([
    {"방법": "기준 (3모델 균등)", "개발 CV": mf(y_dev, pred_base), "홀드아웃": mf(y_hold, hold_base)},
    {"방법": "2단계 판정", "개발 CV": mf(y_dev, pred_new), "홀드아웃": mf(y_hold, hold_new)},
])
result["개발 CV 변화"] = result["개발 CV"] - result.loc[0, "개발 CV"]
result["홀드아웃 변화"] = result["홀드아웃"] - result.loc[0, "홀드아웃"]
print(result.round(4).to_string(index=False))

four = [classes.index(c) for c in ("KIPAN", "KIRC", "GBMLGG", "LGG")]
per_class = pd.DataFrame({
    "기준": f1_score(y_dev, pred_base, average=None)[four],
    "2단계 판정": f1_score(y_dev, pred_new, average=None)[four]}, index=["KIPAN", "KIRC", "GBMLGG", "LGG"])
print("\n개발 CV 암종별 F1")
print(per_class.round(4).to_string())

         방법  개발 CV   홀드아웃  개발 CV 변화  홀드아웃 변화
기준 (3모델 균등) 0.5291 0.5138    0.0000   0.0000
     2단계 판정 0.5275 0.5159   -0.0016   0.0022

개발 CV 암종별 F1
            기준  2단계 판정
KIPAN   0.3779  0.3640
KIRC    0.4392  0.4461
GBMLGG  0.4110  0.4338
LGG     0.5859  0.5767


## 6. 쌍둥이 분리 fold 확인 (개발셋, LR E 단독)
- 이유: 팀 공통 fold는 쌍둥이를 같은 fold에 묶어서, 쌍둥이가 학습과 평가로 갈린 경우의 손실이 CV에 나타나지 않음
- 방법: 쌍둥이 345쌍을 서로 다른 fold에 넣고, 같은 LR(E, C=0.1)로 기준과 2단계 판정 비교
- 3모델 중 LR만 재학습 (XGBoost·LightGBM은 이 조건으로 확인하지 않음)

In [7]:
# 쌍둥이 그룹과 분리 fold (개발셋만 사용)
groups = twin_groups(X_dev_raw, y_dev, classes=le.classes_)
twin = pd.Series(groups).map(pd.Series(groups).value_counts()).to_numpy() >= 2

fold_id = np.zeros(len(y_dev), dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(F_dev, y_dev)):
    fold_id[va] = k
for gid in np.unique(groups[twin]):
    i, j = np.flatnonzero(groups == gid)
    if fold_id[i] == fold_id[j]:              # 같은 fold에 있으면 한 명을 다음 fold로 이동
        fold_id[j] = (fold_id[j] + 1) % 5
split_folds = [(np.flatnonzero(fold_id != k), np.flatnonzero(fold_id == k)) for k in range(5)]

# 피처 E (행 안에서만 계산) + LR (팀 설정 C=0.1, balanced)
E_dev = csr_matrix(pd.concat([
    binarize_mutations(X_dev_raw), np.log1p(mutation_count(X_dev_raw)), np.log1p(mutation_type_counts(X_dev_raw)),
    np.log1p(amino_acid_pair_counts(X_dev_raw)), gene_truncation_flags(X_dev_raw)], axis=1).to_numpy(np.float32))

rows = []
for name, folds in [("팀 공통 fold (쌍둥이 같은 fold)", folds_dev), ("쌍둥이 분리 fold", split_folds)]:
    p_base, p_new = np.zeros(len(y_dev), dtype=int), np.zeros(len(y_dev), dtype=int)
    for tr, va in folds:
        lr = LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000, random_state=SEED).fit(E_dev[tr], y_dev[tr])
        proba = lr.predict_proba(E_dev[va])
        p_base[va] = proba.argmax(1)
        p_new[va] = decode(proba, stage2_proba(F_dev, y_dev, tr, F_dev[va], classes), classes)
    rows.append({"fold": name, "기준": mf(y_dev, p_base), "2단계 판정": mf(y_dev, p_new),
                 "쌍둥이 정확도 (기준)": (p_base == y_dev)[twin].mean(), "쌍둥이 정확도 (2단계)": (p_new == y_dev)[twin].mean()})
twin_table = pd.DataFrame(rows)
twin_table["변화"] = twin_table["2단계 판정"] - twin_table["기준"]
print("쌍둥이 환자:", int(twin.sum()), "/", len(y_dev))
print(twin_table.round(4).to_string(index=False))

쌍둥이 환자: 690 / 4960
                   fold     기준  2단계 판정  쌍둥이 정확도 (기준)  쌍둥이 정확도 (2단계)     변화
팀 공통 fold (쌍둥이 같은 fold) 0.4849  0.4860        0.3681         0.4319 0.0011
            쌍둥이 분리 fold 0.4598  0.4853        0.1319         0.4507 0.0255


## 7. 시험 제출 파일 (저장된 test 예측, submissions/)
- LB 시험용. 채택되면 최종 제출 파일은 submit/final_solution.ipynb(Colab CPU) 출력으로 다시 생성

In [8]:
# 2단계 모델은 train 전체로 학습, test는 행별 변환 후 예측에만 사용
F_test = simple_features(test[X.columns])
s2_test = stage2_proba(F, y, np.arange(len(y)), F_test, classes)
make_submission(test, le.inverse_transform(decode(test_proba, s2_test, classes)),
                "submission_04g_pair_two_stage_1112.csv")

저장: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/submissions/submission_04g_pair_two_stage_1112.csv (2546행)


,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,STES
2,TEST_0002,THCA
3,TEST_0003,LGG
4,TEST_0004,HNSC
...,...,...
2541,TEST_2541,COAD
2542,TEST_2542,PAAD
2543,TEST_2543,UCEC
2544,TEST_2544,LAML


### 8. 결론 (Mac 실행 기준)
- 2단계 판정: 개발 CV −0.0016 (fold 1/5 상승), 홀드아웃 +0.0022 → 판단 기준(OOF +0.005) 미달
- 쌍둥이 분리 fold (LR E 단독): 0.4598 → 0.4853 (+0.0255), 쌍둥이 정확도 0.1319 → 0.4507
- 쌍둥이가 같은 fold에 있는 조건에서는 변화가 작고, 갈린 조건에서만 회복됨
- 한계: XGBoost·LightGBM은 쌍둥이 분리 fold로 재학습하지 않음. test에서 해당되는 행 수는 확인하지 않음 (test 통계 미사용)